In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(".env")  # carga tu contraseña u otros secretos

USER = "root"
PASSWORD = os.getenv("Key")
HOST = "localhost"
PORT = "3306"
DB_NAME = "sakila"

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")

In [ ]:
# Funcion rentals month
def rentals_month(engine, month, year):
    """
    Recupera todos los alquileres del mes y año especificados.
    """
    query = f"""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = {month}
          AND YEAR(rental_date) = {year};
    """
    df = pd.read_sql(query, engine)
    return df


In [ ]:
# Funcion rental_count_month
def rental_count_month(df, month, year):
    """
    Devuelve un DataFrame con el número total de alquileres por cliente.
    """
    col_name = f"alquileres_{month:02d}_{year}"

    df_counts = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=col_name)
    )

    return df_counts


In [ ]:
# Funcion compare_rentals
def compare_rentals(df1, df2):
    """
    Combina dos DataFrames de conteo de alquileres por cliente
    y calcula la diferencia entre ambos meses.
    """
    df_merged = pd.merge(df1, df2, on="customer_id", how="outer").fillna(0)

    col1 = df1.columns[1]
    col2 = df2.columns[1]

    df_merged["diferencia"] = df_merged[col2] - df_merged[col1]

    return df_merged


In [ ]:
# Check
df_may = rentals_month(engine, 5, 2005)
df_jun = rentals_month(engine, 6, 2005)

count_may = rental_count_month(df_may, 5, 2005)
count_jun = rental_count_month(df_jun, 6, 2005)

comparison = compare_rentals(count_may, count_jun)
comparison.head()


,customer_id,alquileres_05_2005,alquileres_06_2005,diferencia
0,1,2.0,7.0,5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,2.0
3,4,0.0,6.0,6.0
4,5,3.0,5.0,2.0
